In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

In [0]:
%sql
DROP TABLE IF EXISTS default.my_csv_table;

Datei einlesen und als Tabelle abspeichern

In [0]:
# 1) Einlesen aus dem Volume
df = (spark.read
    .format("csv")
    .option("header", "true")
    .option("sep", ";")
    .option("inferSchema", "true")
    .load("/Volumes/workspace/default/volume/sample_data.csv"))

# 2) Speichern als persistente Delta-Tabelle im Unity Catalog
(df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.my_csv_table"))

In [0]:
%sql
SELECT * FROM default.my_csv_table

Daten anhängen

In [0]:
df = (spark.read
    .format("csv")
    .option("header", "true")
    .option("sep", ";")
    .option("inferSchema", "true") 
    .load("/Volumes/workspace/default/volume/sample_data.csv"))

(df.write
    .format("delta")
    .mode("append")
    .saveAsTable("workspace.default.my_csv_table"))

In [0]:
%sql
SELECT * FROM default.my_csv_table

Merge

In [0]:
# Ausgangslage: wir haben eine Tabelle
# # 1) Einlesen aus dem Volume
df = (spark.read
    .format("csv")
    .option("header", "true")
    .option("sep", ";")
    .option("inferSchema", "true")
    .load("/Volumes/workspace/default/volume/sample_data.csv"))

# 2) Speichern als persistente Delta-Tabelle im Unity Catalog
(df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.my_csv_table"))

In [0]:
%sql
SELECT * FROM default.my_csv_table

In [0]:
from delta.tables import DeltaTable

# 1. Neue CSV einlesen mit Schema-Erkennung
updates_df = (spark.read
    .format("csv")
    .option("header", "true")
    .option("sep", ";")
    .option("inferSchema", "true")
    .load("/Volumes/workspace/default/volume/sample_data2.csv"))

In [0]:
# Data Exploration vom neune CSV
display(updates_df)

In [0]:
 # 2. Delta-Tabelle laden
target_table = DeltaTable.forName(spark, "default.my_csv_table")

In [0]:
# 3. MERGE ausführen
# 'target.id = updates.id' 
# funktioniert nur, wenn beide IDs denselben Datentyp haben!
(target_table.alias("target")
  .merge(
    updates_df.alias("updates"),
    "target.id = updates.id" 
  )
  .whenMatchedUpdateAll()     # Wenn ID existiert -> Zeile aktualisieren
  .whenNotMatchedInsertAll()  # Wenn ID neu ist -> Zeile hinzufügen
  .execute())

print("Upsert erfolgreich abgeschlossen.")

In [0]:
%sql
SELECT * FROM default.my_csv_table ORDER BY id

-- Alice hat UPDATE vom Attribut "age" erhalten
-- Bob unverändert
-- Charlie unverändert
-- Hans INSERT

#Nested JSON Data

In [0]:
%sql
drop table if exists default.nestedjson

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

# 1. Manuelles Schema definieren (Best Practice für Produktion)
nested_schema = StructType([
    StructField("email", StringType(), True),
    StructField("gpa", DoubleType(), True),
    StructField("stu", StringType(), True),
    StructField("profile", StructType([
        StructField("first_name", StringType(), True),
        StructField("last_name", StringType(), True),
        StructField("gender", StringType(), True),
        StructField("address", StructType([
            StructField("street", StringType(), True),
            StructField("city", StringType(), True),
            StructField("country", StringType(), True)
        ]))
    ]))
])

# 2. JSON mit dem Schema laden
# Hinweis: Das Schema sorgt für Sicherheit, falls Felder im JSON fehlen oder falsch sind.
df_nested = (spark.read
    .schema(nested_schema)
    .json("/Volumes/workspace/default/volume/nested_json.json"))

# 3. Als Delta-Tabelle im Unity Catalog speichern (3-Level Namespace)
(df_nested.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.nested_json_table"))

In [0]:
%sql
SELECT * FROM default.nestedjson

In [0]:
%sql
SELECT
  stu,
  profile.first_name AS first_name,
  profile.last_name AS last_name,
  profile.address.country AS country
FROM default.nestedjson

# Aufräumen

In [0]:
%sql
DROP TABLE IF EXISTS default.nestedjson;
DROP TABLE IF EXISTS default.my_csv_table;
